# Robots Vote: Do Robots Influence Your Vote?
**Experiment support code** <br>
This code supports the experiment, allowing us to control the behavior of the robots. 

## Getting Started...
Imports and auxiliary code

### Robots

In [5]:
import asyncio
import soundfile as sf
import numpy as np
from reachy_mini import ReachyMini
from dash.robot import DashRobot, discover_and_connect
import time
import random
from reachy_mini.utils import create_head_pose
import os
import wave
from reachy_mini.motion.recorded_move import RecordedMoves

EMOTIONS_DATASET = "pollen-robotics/reachy-mini-emotions-library"
emotions = RecordedMoves(EMOTIONS_DATASET)


class Robot:
    def __init__(self, name, typer):
        self.name = name
        self.type = typer
        self.device = None
        self.audio_path=None
        self.audio_slot=None
        self.state = "idle"
        self.idle_animations_reachy = ["curious1", "serenity1", "cheerful1"]
        self.listen_animations_reachy = ["proud1", "understanding2", "attentive2"]
        
    async def connect(self):
        try:
            if self.type == "dash": 
                
                robo = await discover_and_connect([self.name])
                if robo:
                    self.device = robo
            
                
            elif self.type == "reachy":
                print(f"[{self.name}] A procurar Reachy...")
                #self.device = ReachyMini(id=self.name)
                if self.name == "Robot1":
                    self.device = ReachyMini(host="reachy-mini-one.local")
                elif self.name == "Robot2":
                    self.device = ReachyMini(host="reachy-mini-two.local")
                else:
                    self.device = ReachyMini()
                self.device.media.start_playing()
                self.device.goto_target(head=create_head_pose(), duration=0.2)
                
            else:
                print("RobotSim " + self.name + " is connected!")
            print(f"✅ {self.name} connected")
        except Exception as e:
            print(f"❌ Robot {self.name} falied to connect: {e}")

            

    def is_connected(self):
        return self.device is not None or (self.type != "dash" and self.type != "reachy")
    
    def change_audio(self, audio):
        if self.type == "dash":
            self.audio_slot=audio
        elif self.type == "reachy":
            self.audio_path=audio

    async def animete_and_talk(self,duration):
        if self.type == "dash": 
            await self.device.say(self.audio_slot) 
            end = time.time() + duration
            while time.time() < end:
                for brilho in [0, 75, 150, 75]:
                        print("olho")
                        await self.device.eye_brightness(brilho)
                        await asyncio.sleep(0.15)
        if self.type == "reachy":
            wav_path = os.path.abspath(self.audio_path)
            data, samplerate = sf.read(wav_path)
            
            if len(data.shape) > 1:
                data = data.mean(axis=1)
        
            with wave.open(wav_path, "rb") as wf:
                # Duração real do ficheiro
                wav_duration = wf.getnframes() / wf.getframerate()
        
            print(f"A iniciar: {wav_path} (Duração: {wav_duration:.2f}s)")
            
            end = time.time() + wav_duration + 2
                
            
            self.device.media.play_sound(wav_path)
            talk_task = asyncio.create_task(self.device.async_play_move(emotions.get("attentive1"), initial_goto_duration=0.1, sound=False))
            while time.time() < end:
                if(talk_task.done()):
                    talk_task = asyncio.create_task(self.device.async_play_move(emotions.get("attentive1"), initial_goto_duration=0.1, sound=False))
                await asyncio.sleep(0.5)
        
            print("Done!")
        
            self.device.cancel_move() # Interrompe qualquer movimento residual do duration
            self.device.goto_target(head=create_head_pose(), duration=0.2)
            print("Concluído.")

        else: 
            end = time.time() + duration
            while time.time() < end:
                print("RobotSim " + self.name + " is talking!")
                await asyncio.sleep(1.5)
                


    async def idle_animation(self, duration):
        end = time.time() + duration
        yaw = 0
        
        if self.type == "dash": 
            while time.time() < end:
                print(self.device.get_head_yaw())
                yaw = min(20,max(-20,yaw+random.randint(-3, 3)))
                await self.device.head_yaw(yaw)
                await asyncio.sleep(random.uniform(0.5, 2.0))
            
        elif self.type == "reachy":    
            idle_task = asyncio.create_task(self.device.async_play_move(emotions.get(random.choice(self.idle_animations_reachy)), initial_goto_duration=0.5, sound=False))
            while time.time() < end:
                if(idle_task.done()):
                    print("idle")
                    idle_task = asyncio.create_task(self.device.async_play_move(emotions.get(random.choice(self.idle_animations_reachy)), initial_goto_duration=0.5, sound=False))
                await asyncio.sleep(1.5)

            self.device.cancel_move() # Interrompe qualquer movimento residual do duration
            self.device.goto_target(head=create_head_pose(), duration=0.2)
            
        else: 
            print("RobotSim " + self.name + " is idling!")
            await asyncio.sleep(random.uniform(2.5, 4.0))

            

    async def listening_animation(self, duration):
        end = time.time() + duration
        
        
        if self.type == "dash":  
            while time.time() < end:
                for brilho in ["#000000", "#ffffff", "#000000", "#ffffff", "#000000", "#ffffff", "#000000", "#ffffff",]:
                    await self.device.left_ear_color(brilho)
                    await self.device.right_ear_color(brilho)
                    await asyncio.sleep(1.0)
                
        elif self.type == "reachy": 
            listen_task = asyncio.create_task(self.device.async_play_move(emotions.get(random.choice(self.listen_animations_reachy)), initial_goto_duration=0.5, sound=False))

            while time.time() < end:
                if(listen_task.done()):
                    print("listen")
                    listen_task = asyncio.create_task(self.device.async_play_move(emotions.get(random.choice(self.listen_animations_reachy)), initial_goto_duration=0.5, sound=False))
            
                await asyncio.sleep(1.5)

            self.device.cancel_move() # Interrompe qualquer movimento residual do duration
            self.device.goto_target(head=create_head_pose(), duration=0.2)
            

        else:
            print("RobotSim " + self.name + " is listening!")
            await asyncio.sleep(1.5)
        

    async def disconnect(self):
        if self.type == "dash": 
            await self.device.disconnect()
            print("Dash " + self.name + " disconnected!")
        elif self.type == "reachy":
            self.device = None
            print("Reachy " + self.name + " disconnected!")
        else:
            print("RobotSim " + self.name + " disconnected!")

    def is_speaking(self):
        return self.state == "speak"
    
    def set_state(self, state):
        self.state = state
        

    async def run_loop(self):
        print(self.state)
        while True:
            match self.state:
                case "idle":
                    task1 = asyncio.create_task(self.idle_animation(10))
                    while (not task1.done()):
                        await asyncio.sleep(1)
                case "listen":
                    print("Listening")
                    task1 = asyncio.create_task(self.listening_animation(10))
                    while not task1.done():
                        await asyncio.sleep(1)
                case "speak":
                    task2 = asyncio.create_task(self.animete_and_talk(15))
                    while (not task2.done()):
                        await asyncio.sleep(1)
                    self.state = "listen"
                    
                case "stop":
                    self.state = "idle"
                    break
                case _:
                    return "Something's wrong with "+self.name
        


In [4]:
reachy = Robot("Robot7","dash")
reachy.change_audio("my1")
await reachy.connect()
asyncio.create_task(reachy.run_loop())
await asyncio.sleep(20)
reachy.set_state("listen")
await asyncio.sleep(20)
reachy.set_state("speak")
await asyncio.sleep(10)
reachy.set_state("stop")

Connected to C1:B1:3B:AC:9A:B7
✅ Robot7 connected
idle
202
247
4
16
26
26
28
13
21
20
9
252
252
252
243
243
243
247
235
Listening
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
olho
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
RobotSim Robot7 is talking!
Listening
Listening
Listening
Listening
Listening
Listening
Listening
Listening
Listening


In [6]:
reachy = Robot("Robot1","reachy")
reachy.change_audio("test2.wav")
await reachy.connect()
asyncio.create_task(reachy.run_loop())
await asyncio.sleep(20)
reachy.set_state("listen")
await asyncio.sleep(20)
reachy.set_state("speak")
await asyncio.sleep(10)
reachy.set_state("stop")


INFO:reachy_mini.reachy_mini:Auto connection: localhost attempt failed (Failed to connect to ws://localhost:8000/ws/sdk: [Errno 111] Connection refused). Trying remote host reachy-mini-one.local.
INFO:reachy_mini.reachy_mini:Connection mode selected: network


[Robot1] A procurar Reachy...


INFO:reachy_mini.reachy_mini:No local IPC endpoint. Using WebRTC backend for streaming.
INFO:reachy_mini.media.media_manager:Using WebRTC streaming backend.
ERROR:reachy_mini.media.audio_control_utils:No Reachy Mini Audio USB device found!
INFO:reachy_mini.media.webrtc_client_gstreamer:GstWebRTCClient initialized (bidirectional audio support)


✅ Robot1 connected
idle


INFO:reachy_mini.reachy_mini:Move cancellation requested
INFO:reachy_mini.reachy_mini:Move cancelled, stopping playback


...

INFO:reachy_mini.reachy_mini:Move cancellation requested


Done!


INFO:reachy_mini.reachy_mini:Move cancelled, stopping playback


Concluído.
Listening
listen
listen


INFO:reachy_mini.reachy_mini:Move cancellation requested


In [1]:
import speech_recognition as sr
import whisper
import torch
from sentence_transformers import SentenceTransformer, util
import time
import os
import numpy as np


def processar_com_whisper(statement_id, statements, robots):
    rec = sr.Recognizer()
    
    rec.pause_threshold = 2.0  # Espera 3s de silêncio antes de processar
    rec.dynamic_energy_threshold = True 
    
    mic = sr.Microphone()

    with mic as source:
        print("\n[CALIBRAÇÃO] Silêncio na sala...")
        rec.adjust_for_ambient_noise(source, duration=2)
        print("SISTEMA ATIVO (Whisper Local). O participante pode entrar na sala...")
        
        while True:
            try:
                # Ouve o áudio e guarda em memória
                print("\nÀ escuta... O statement pode ser lido a qualquer momento...")
                audio = rec.listen(source, timeout=None, phrase_time_limit=15)
                
                print("-> Áudio capturado. Whisper a processar...")
                
                for robot in robots.values():
                    robot.set_state("listen")

                audio_data = audio.get_raw_data(convert_rate=16000, convert_width=2)
                audio_np = np.frombuffer(audio_data, dtype=np.int16).astype(np.float32) / 32768.0
                
                # Transcreve diretamente do array (o Whisper suporta isto)
                result = model_stt.transcribe(audio_np, language="pt")
            
                texto = result["text"].lower().strip()
                
                if len(texto.split()) < 3:
                    continue

                print(f"O Whisper ouviu: '{texto}'")

                # Comparação Semântica
                emb_ouvido = model_sim.encode(texto, convert_to_tensor=True)
                cos_sim = util.cos_sim(emb_ouvido, embeddings_ref)[0]
                idx = cos_sim.argmax().item()
                conf = cos_sim[idx].item()

                print(statement_id)
                print(f"Confiança Semântica: {conf:.2f}")
                if conf > 0.65 and (idx + 1) == int(statement_id):
                    print(f"✅ SUCESSO: Statement {idx + 1}: {statements['robot_related'][idx]}")                   
                    break


            except Exception as e:
                print(f"Erro: {e}")


### Articles

In [3]:
from PIL import Image, ImageDraw, ImageFont
import threading
from http.server import SimpleHTTPRequestHandler, HTTPServer

# =========================
# CONFIG
# =========================



# =========================
# CENTER CROP (IMPORTANTE)
# =========================
def crop_center(img, target_w, target_h):
    img_ratio = img.width / img.height
    target_ratio = target_w / target_h

    # cortar largura
    if img_ratio > target_ratio:
        new_width = int(img.height * target_ratio)
        offset = (img.width - new_width) // 2
        img = img.crop((offset, 0, offset + new_width, img.height))
    else:
        new_height = int(img.width / target_ratio)
        offset = (img.height - new_height) // 2
        img = img.crop((0, offset, img.width, offset + new_height))

    return img.resize((target_w, target_h))


# =========================
# WRAP TEXTO
# =========================
def wrap_text(text, font, max_width, draw):
    lines = []
    words = text.split()

    while words:
        line = ""
        while words:
            test_line = line + words[0] + " "
            bbox = draw.textbbox((0, 0), test_line, font=font)
            w = bbox[2] - bbox[0]

            if w <= max_width:
                line = test_line
                words.pop(0)
            else:
                break

        lines.append(line.strip())

    return lines


def criar_card(titulo, quote, imagem_path, output):

    WIDTH, HEIGHT = 540, 540
    
    img = Image.open(imagem_path).convert("RGB")
    img = crop_center(img, WIDTH, HEIGHT)

    # escurecer ligeiramente (melhor leitura)
    black = Image.new("RGB", (WIDTH, HEIGHT), (0, 0, 0))
    img = Image.blend(img, black, 0.8)

    draw = ImageDraw.Draw(img)

    titulo_font = ImageFont.truetype("./Lato/Lato-Bold.ttf", 30)
    quote_font = ImageFont.truetype("./Lato/Lato-Regular.ttf", 20)

    max_width = 500

    titulo_lines = wrap_text(titulo, titulo_font, max_width, draw)

    y = 50

    for line in titulo_lines:
        bbox = draw.textbbox((0, 0), line, font=titulo_font)
        text_width = bbox[2] - bbox[0]

        x = (WIDTH - text_width) / 2

        draw.text((x, y), line, font=titulo_font, fill="white")
        y += 50

    quote_text = f"“{quote}”"
    quote_lines = wrap_text(quote_text, quote_font, max_width, draw)

    line_height = 25
    total_height = len(quote_lines) * line_height

    y_quote = HEIGHT - total_height - 90

    for line in quote_lines:
        bbox = draw.textbbox((0, 0), line, font=quote_font)
        text_width = bbox[2] - bbox[0]

        x = (WIDTH - text_width) / 2

        draw.text((x, y_quote), line, font=quote_font, fill="white")
        y_quote += line_height

    img.save(output)
    print(f"✔ Guardado: {output}")

### Virtual agents

In [38]:
_model = None
def get_model(statements):
    global _model
    if _model is None:
        print("A carregar Sentence-BERT...")
        model_sim = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        embeddings_ref = model_sim.encode(statements[topic], convert_to_tensor=True)
    return model_sim, embeddings_ref
 
 
def find_best_statement(prompt, statements):
    print("Prompt: " + prompt)

    print(repr(prompt))
    print(len(prompt))
    model, embeddings_ref = get_model(statements)
    emb_prompt = model.encode(prompt, convert_to_tensor=True)

    cos_sim = util.cos_sim(emb_prompt, embeddings_ref)[0]
    idx = cos_sim.argmax().item()
    conf = cos_sim[idx].item()
    
    print(f"  → Statement: {idx} (score={conf:.2f})")
    if(conf > 0.6):
        return idx
    return 7
    
 
 
def get_responses(json_data, topic, statement_id, condition):
    """Devolve lista de 4 textos, um por agente, pela ordem do json."""
    
    block = json_data[topic][str("virtuals")][str(statement_id)]["participant_disagree"][condition]
 
    texts = []
    for entry in block.values():
        # tenta campos comuns — ajusta ao nome real no teu JSON
        
        texts.append(entry["quote"])
 
    while len(texts) < 4:
        texts.append("(sem resposta)")
    return texts[:4]


def make_handler(statements, json_data, topic, condition):
    class Handler(SimpleHTTPRequestHandler):
 
        def log_message(self, fmt, *args):
            pass  # silenciar logs

        def do_GET(self):
            with open("chat_virtual.html", "rb") as f:
                data = f.read()
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.end_headers()
            self.wfile.write(data)

 
        def do_POST(self):
            if self.path == "/ask":
                length = int(self.headers.get("Content-Length", 0))
                body = json.loads(self.rfile.read(length))
                prompt = body.get("prompt", "").strip()
 
                sid = find_best_statement(prompt, statements)
                texts = get_responses(json_data, topic, sid, condition)
 
                resp = json.dumps({
                    "statement_id":   sid,
                    "statement_text": statements.get(sid, ""),
                    "responses": {
                        "claude":  texts[0],
                        "gpt":     texts[1],
                        "gemini":  texts[2],
                        "mistral": texts[3],
                    }
                }).encode()
 
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.send_header("Access-Control-Allow-Origin", "*")
                self.end_headers()
                self.wfile.write(resp)
            else:
                self.send_response(404)
                self.end_headers()
 
        def do_OPTIONS(self):
            self.send_response(200)
            self.send_header("Access-Control-Allow-Origin", "*")
            self.send_header("Access-Control-Allow-Methods", "POST, GET, OPTIONS")
            self.send_header("Access-Control-Allow-Headers", "Content-Type")
            self.end_headers()
 
    return Handler



## Experiment

### Step 1 - Randomization of the experiment
Run this code to randomize the experience

In [40]:
import random
import json

statements = {
    "economic":[],
    "social":[],
    "robot_related":[
    "Os robôs são benéficos para a sociedade, porque ajudam as pessoas.",
    "Atribuir tarefas de rotina aos robôs permite que as pessoas se dediquem a tarefas mais significativas.",
    "Receio que os robôs incentivem uma menor interação entre os seres humanos.",
    "As tarefas perigosas devem ser atribuídas, em primeiro lugar, aos robôs.",
    "A utilização generalizada de robôs vai tirar empregos às pessoas.",
    "A utilização não regulamentada da robótica pode levar a mudanças sociais.",
    "Acabou esta discussão, obrigado a todos."]
}

#topics = ["economic", "social", "robot_related"]
topics = [ "robot_related"]
with open('matrix.json', encoding='utf-8') as f:
    json_data = json.load(f)

with open('participant.json') as f:
    participant_data = json.load(f)


topic = random.choice(topics)
    
print("Tópico: "+ topic)

        
dic_aux = {
    "robots": ["all_disagree", "half_disagree"],
    "virtuals": ["all_disagree", "half_disagree"],
    "articles": ["all_disagree", "half_disagree"]
}
    
ids = [1, 2, 3, 4, 5, 6]
random.shuffle(ids)  # mistura os IDs
    
resultado = {}
i = 0
    
for key, values in dic_aux.items():
    resultado[key] = {}
    for v in values:
        resultado[key][v] = ids[i]
        i += 1
            
conditions = []
    
for categoria, condicoes in resultado.items():
    for condicao, statement_id in condicoes.items():
            
        conditions.append([str(statement_id),categoria,condicao])

random.shuffle(conditions)
print("\nOs statements vão ser discutidos na seguinte ordem: ")
for statement_id,agents,condition in conditions:
    print(f"O statement {statement_id} vai ser discutido com {agents} na condição {condition}.")
    


Tópico: robot_related

Os statements vão ser discutidos na seguinte ordem: 
O statement 3 vai ser discutido com virtuals na condição half_disagree.
O statement 5 vai ser discutido com robots na condição all_disagree.
O statement 6 vai ser discutido com articles na condição all_disagree.
O statement 2 vai ser discutido com virtuals na condição all_disagree.
O statement 1 vai ser discutido com articles na condição half_disagree.
O statement 4 vai ser discutido com robots na condição half_disagree.


## 2 - Experiment Rounds

In [43]:

i = 1
for statement_id,agents,condition in conditions:
    print(f"{i} - O statement {statement_id} vai ser discutido com {agents} na condição {condition}.")
    i+=1
r = input("\nEscolhe qual das rondas queres fazer:")


print("\nVamos começar...")
statement_id,agents,condition = conditions[int(r)-1]


print("\nStatement: " + statement_id)

        
if agents == "robots":
    print("A carregar Whisper (base)...")
    model_stt = whisper.load_model("small") # Modelo de voz (corre no teu PC)
    print("A carregar Sentence-BERT...")
    model_sim = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    embeddings_ref = model_sim.encode(statements[topic], convert_to_tensor=True)
    print("\nColocar os robos seguintes na mesa: ")
    robots = {}

    robots["Robot1"] = Robot("Robot1","")
    robots["Robot2"] = Robot("Robot2","reachy")
    robots["Robot7"] = Robot("Robot7","")
    robots["Robot8"] = Robot("Robot8","")
    
    #Descomentar quando for para usar os robos reais
    #for robot in json_data[topic][str(agents)][statement_id]["participant_disagree"][condition].keys():
        #print(robot)
        #typer = json_data[topic][str(agents)][statement_id]["participant_disagree"][condition][robot]["type"]
        #print(typer)
        #robots[robot] = Robot(robot,"")

    #robots["Robot2"]. colocar o audio
    

    input("\nCarrega Enter para continuar, quando os robos estiverem a postos e ligados...")

    print("\nA conectar os robos...")

    while True:
        for robot in robots.values():
            if not robot.is_connected():
                await robot.connect()
        all_connected = all(robot.is_connected() for robot in robots.values())
        if all_connected:
            break

    for robot in robots.values():
        print(str(robot.name) + "-" + str(robot.is_connected()))
    
    print("\nRobos connectados")
    print("\nA iniciar os robos...")
    for robot in robots.values():
        asyncio.create_task(robot.run_loop())

    print("\nRobos iniciados... O participante pode entrar na sala.")
    
    task = asyncio.create_task(asyncio.to_thread(processar_com_whisper, statement_id, statements, robots))
    while not task.done():
        await asyncio.sleep(1)
    

    #Robos a responder
    for robot in robots.values():
        robot.set_state("speak")
        while robot.is_speaking():
            await asyncio.sleep(1)

    print("\nVEz do participante... a esperar que o investigador acabe")
                
        
    task = asyncio.create_task(asyncio.to_thread(processar_com_whisper, statement_id, statements, robots))
    while not task.done():
        await asyncio.sleep(1)

    for robot in robots.values():
        robot.set_state("idle")

    input("\nCarrega Enter para finalizar...")

    for robot in robots.values():
        robot.set_state("stop")

if agents == "articles":

    print("\nA criar as imagens...")
    print(json_data[topic][str(agents)].keys())
    for i, article in enumerate(json_data[topic][str(agents)][statement_id]["participant_disagree"][condition].keys()):

        criar_card(json_data[topic][str(agents)][statement_id]["participant_disagree"][condition][article]["title"], json_data[topic][str(agents)][statement_id]["participant_disagree"][condition][article]["quote"], ".\\articles\\background\\"+topic+".jpg", f".\\articles\\card_{i+1}.png")
    
    

    html = """
    <!DOCTYPE html>
    <html lang="pt">
    <head>
        <meta charset="UTF-8">
        <title>Notícias</title>
    </head>
    <style>
    body {
        margin: 0;
        background: #111;
        display: flex;
        justify-content: center;
        align-items: center;
    }
    .grid {
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 20px;
        width: 90vw;
        max-width: 900px;
    }
    img {
        width: 100%;
        height: auto;
        border-radius: 12px;
    }
    </style>
    </head>
    <body>
    
    <div class="grid">
        <img src=".\\articles\\card_1.png">
        <img src=".\\articles\\card_2.png">
        <img src=".\\articles\\card_3.png">
        <img src=".\\articles\\card_4.png">
    </div>
    
    </body>
    </html>
    """
    
    with open("index.html", "w") as f:
        f.write(html)

    PORT = 8000
    server = None
    
    
    def start_server():
        global server
        server = HTTPServer(("0.0.0.0", PORT), SimpleHTTPRequestHandler)
        print(f"Servidor a correr em http://0.0.0.0:{PORT}")
        server.serve_forever()
    
    
    # arrancar servidor
    thread = threading.Thread(target=start_server)
    thread.start()
    
    # esperar input
    input("👉 Carrega ENTER para parar o servidor...\n")
    
    # parar servidor
    if server:
        print("A parar servidor...")
        server.shutdown()
        server.server_close()
        print("Servidor parado.")

if agents == "virtuals":

    
    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html)

    PORT = 8000
    server = None
    
    
    def start_server():
        global server
        server = HTTPServer(("0.0.0.0", PORT), make_handler(statements, json_data, topic, condition))
        print(f"Servidor a correr em http://0.0.0.0:{PORT}")
        server.serve_forever()
    
    
    # arrancar servidor
    thread = threading.Thread(target=start_server)
    thread.start()



    # esperar input
    input("👉 Carrega ENTER para parar o servidor...\n")
    
    # parar servidor
    if server:
        print("A parar servidor...")
        server.shutdown()
        server.server_close()
        print("Servidor parado.")

            

            
                

            
            
            

    

1 - O statement 3 vai ser discutido com virtuals na condição half_disagree.
2 - O statement 5 vai ser discutido com robots na condição all_disagree.
3 - O statement 6 vai ser discutido com articles na condição all_disagree.
4 - O statement 2 vai ser discutido com virtuals na condição all_disagree.
5 - O statement 1 vai ser discutido com articles na condição half_disagree.
6 - O statement 4 vai ser discutido com robots na condição half_disagree.



Escolhe qual das rondas queres fazer: 1



Vamos começar...

Statement: 3


👉 Carrega ENTER para parar o servidor...
 


Servidor a correr em http://0.0.0.0:8000


## Economic

### Question 1 - ""

#### Total opposition

In [ ]:

for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my1") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my1") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 2 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my2") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my2") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 3 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my3") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my3") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

## Social

### Question 1 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my4") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my4") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 2 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my5") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my5") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 3 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my6") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my6") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

## Robot-Related

### Question 1 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my7") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my7") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 2 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my8") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my8") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 3 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my9") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my9") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

## Partial opposition

### Economic

In [ ]:
if "Fernanda" in robos:
    # 1. Inicia o som
    await robos["Fernanda"].say("my2") 
    # 2. Roda a animação sem travar o código (create_task)
    task = asyncio.create_task(animar_olho(robos["Fernanda"], duracao=10.0))
    # 3. Espera a animação terminar antes de você ir para a próxima célula
    await task

### Social

### Robot-Related

### Participant talking

In [ ]:
if "Fernanda" in robos:
    # 1. Inicia o som
    await robos["Fernanda"].say("my2") 
    # 2. Roda a animação sem travar o código (create_task)
    task = asyncio.create_task(mode_listening(robos["Fernanda"], duracao=10.0))
    # 3. Espera a animação terminar antes de você ir para a próxima célula
    await task

## Total opposition


### Economic

### Social

### Robot-Related

### Participant talking

In [1]:
import asyncio
from dash.robot import DashRobot, discover_and_connect
import time

async def main():
    robot = await discover_and_connect(["Fernanda"])
    print(robot)
    if robot:
        print(f"Connected to {robot.address}")
        # Example command to move Dash
        await robot.say("my1")  # Move forward 100mm at 100mm/s
        

if __name__ == "__main__":
    await main()

Connected to CD:2C:91:2D:0D:0C
Connected to CD:2C:91:2D:0D:0C


In [6]:
robot = await discover_and_connect(["Fernanda"])
await robot.disconnect()

CancelledError: 

In [3]:
import asyncio
from dash.robot import DashRobot, discover_and_connect
import time

# Dicionário global para manter os robôs conectados entre as células
robos = {}

async def conectar_equipe():
    nomes = ["Fernanda"] # Substitua pelos nomes reais
    print(f"Buscando por: {', '.join(nomes)}...")
    
    for nome in nomes:
        try:
            # Tenta conectar a cada um individualmente para melhor controle
            robo = await discover_and_connect([nome])
            if robo:
                robos[nome] = robo
                print(f"✅ {nome} conectado em {robo.address}")
        except Exception as e:
            print(f"❌ Falha ao conectar em {nome}: {e}")

# Executa a conexão
await conectar_equipe()

Buscando por: Fernanda...
Connected to CD:2C:91:2D:0D:0C
✅ Fernanda conectado em CD:2C:91:2D:0D:0C


In [21]:
if "Fernanda" in robos:
    # 1. Inicia o som
    await robos["Fernanda"].say("my2") 
    # 2. Roda a animação sem travar o código (create_task)
    task = asyncio.create_task(animar_olho(robos["Fernanda"], duracao=10.0))
    # 3. Espera a animação terminar antes de você ir para a próxima célula
    await task

## Finalize the study

In [4]:
async def disconnect_robots():
    for nome, r in robots.items():
        await r.eye_brightness(0) 
        await r.disconnect()
    print("All robots disconnected.")

await disconnect_robots()

NameError: name 'robots' is not defined